In [ ]:
# Run first on Google Colab
!pip install qutip -q

# §1 Foundations

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Represent qubits and multi-qubit states as QuTiP `Qobj`s and apply gates
- Visualise states on the Bloch sphere
- Build the Hadamard transform and Bell states
- Compute Schmidt decompositions and Schmidt rank


In [ ]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

print(f"QuTiP {qt.__version__}")


---
## Part 1: Foundations

### 1.1 Quantum states and the Born rule

**Four postulates (numerical-analysis translation)**

| Postulate | Linear algebra statement |
|-----------|--------------------------|
| **State space** | A quantum state is a unit vector $|\psi\rangle \in \mathbb{C}^{2^n}$ |
| **Evolution** | Gates are unitary matrices: $|\psi\rangle \mapsto U|\psi\rangle$ |
| **Measurement** | Outcome $k$ has probability $|\langle k|\psi\rangle|^2$ (Born's rule) |
| **Composition** | Joint state lives in $\mathcal{H}_A \otimes \mathcal{H}_B$ |

A **qubit** is a unit vector in $\mathbb{C}^2$.

The **computational basis**: $|0\rangle = (1,0)^\top$, $|1\rangle = (0,1)^\top$.

**General state**: $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$, $|\alpha|^2 + |\beta|^2 = 1$.


In [ ]:
# ── Basis and superposition states ───────────────────────────────────────────
zero = qt.basis(2, 0)          # |0>
one  = qt.basis(2, 1)          # |1>
plus  = (zero + one).unit()    # |+> = (|0>+|1>)/sqrt(2)
minus = (zero - one).unit()    # |-> = (|0>-|1>)/sqrt(2)

print("Ket |0>:", zero.full().flatten())
print("Ket |+>:", plus.full().flatten())

# ── Born rule ─────────────────────────────────────────────────────────────────
theta, phi = np.pi / 3, np.pi / 4
psi = np.cos(theta/2) * zero + np.exp(1j*phi) * np.sin(theta/2) * one

def prob(basis_state, ket):
    """P(|basis_state>) = |<basis_state|ket>|^2"""
    amp = basis_state.dag() * ket   # QuTiP 5 returns complex directly
    return abs(amp)**2

p0 = prob(zero, psi)
p1 = prob(one,  psi)
print(f"\nFor |ψ⟩ with θ=π/3, φ=π/4:")
print(f"  P(|0⟩) = {p0:.4f},  P(|1⟩) = {p1:.4f},  sum = {p0+p1:.6f}")


### 1.2 Standard quantum gates

**Pauli gates:**
$$\mathsf{X} = \begin{pmatrix}0&1\\1&0\end{pmatrix}, \quad
  \mathsf{Y} = \begin{pmatrix}0&-i\\i&0\end{pmatrix}, \quad
  \mathsf{Z} = \begin{pmatrix}1&0\\0&{-1}\end{pmatrix}$$

**Hadamard:**
$\mathsf{H} = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&{-1}\end{pmatrix}$

**Rotation gates:**
$R_y(\theta) = e^{-i(\theta/2)\mathsf{Y}} = \begin{pmatrix}\cos\theta/2 & -\sin\theta/2 \\ \sin\theta/2 & \cos\theta/2\end{pmatrix}$

**Controlled-NOT:** $\mathsf{CNOT}|c\rangle|t\rangle = |c\rangle|c\oplus t\rangle$


In [ ]:
X = qt.sigmax()
Y = qt.sigmay()
Z = qt.sigmaz()
H = qt.gates.hadamard_transform(1)   # Hadamard on 1 qubit
I = qt.qeye(2)

# ── Verify P^2 = I for each Hermitian gate ────────────────────────────────────
print("Verify P² = I:")
for name, P in [("X", X), ("Y", Y), ("Z", Z), ("H", H)]:
    ok = np.allclose((P * P).full(), I.full())
    print(f"  {name}² = I: {ok}")

# ── Verify HZH = X ────────────────────────────────────────────────────────────
print("\nHZH = X:", np.allclose((H * Z * H).full(), X.full()))

# ── Phase rotation gate R_φ = diag(1, e^{iφ}) ────────────────────────────────
def phase_R(phi):
    return qt.Qobj(np.diag([1.0, np.exp(1j * phi)]))

# ── CNOT as projectors ────────────────────────────────────────────────────────
P0 = zero * zero.dag()    # |0><0|
P1 = one  * one.dag()     # |1><1|
CNOT = qt.tensor(P0, I) + qt.tensor(P1, X)
print("\nCNOT matrix:\n", CNOT.full().real.astype(int))


### 1.3 The Bloch sphere

Every single-qubit pure state $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$ corresponds to a point on the unit sphere at angles $(\theta, \phi)$.

Unitary gates are rotations of the Bloch sphere.


In [ ]:
states = [zero, one, plus, minus,
          (zero + 1j*one).unit(),   # |+y>
          (zero - 1j*one).unit()]   # |-y>
labels = ["|0⟩", "|1⟩", "|+⟩", "|-⟩", "|+y⟩", "|-y⟩"]

b = qt.Bloch()
b.add_states(states)
b.zlabel = [r"$|0\rangle$", r"$|1\rangle$"]
b.show()


### Exercise 1.1 — Eigensystem of X and Z

**(a)** Find the eigenvalues and eigenvectors of $\mathsf{X}$ and $\mathsf{Z}$.

**(b)** Show that $\mathsf{H}$ is the change-of-basis matrix between the two eigenbases; verify $\mathsf{HZH} = \mathsf{X}$.


In [ ]:
# YOUR CODE HERE
# Hint: use np.linalg.eigh() or qt.Qobj.eigenstates()

# (a) Eigensystem of X
# vals_X, vecs_X = ...

# (b) Eigensystem of Z
# vals_Z, vecs_Z = ...

# Verify H maps eigenbasis of Z to eigenbasis of X
# ...


In [ ]:
#@title Solution — Exercise 1.1  {display-mode: "form"}
# (a) Eigensystem of X and Z
vals_X, vecs_X = X.eigenstates()
vals_Z, vecs_Z = Z.eigenstates()
print("X eigenvalues:", vals_X, "  eigenvectors:", [v.full().flatten() for v in vecs_X])
print("Z eigenvalues:", vals_Z, "  eigenvectors:", [v.full().flatten() for v in vecs_Z])

# (b) H maps Z-eigenvectors to X-eigenvectors
print("\nH|0⟩ =", (H * zero).full().flatten(), "  compare |+⟩ =", plus.full().flatten())
print("H|1⟩ =", (H * one ).full().flatten(), "  compare |-⟩ =", minus.full().flatten())
print("\nHZH = X:", np.allclose((H * Z * H).full(), X.full()))
# H diagonalises X because its columns are the eigenvectors of X
print("H columns are X-eigenvectors:",
      np.allclose(np.abs(H.full()), np.abs(np.column_stack([v.full().flatten() for v in vecs_X]))))


### 1.4 The Hadamard transform

For $n$ qubits, $\mathsf{H}^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}}\sum_{z\in\{0,1\}^n}(-1)^{x\cdot z}|z\rangle$.

In particular, $\mathsf{H}^{\otimes n}|0^n\rangle = \frac{1}{\sqrt{2^n}}\sum_z |z\rangle$ — a uniform superposition.

This uses only $n$ gates to act on a $2^n$-dimensional state vector, but the result is a superposition — individual amplitudes can only be accessed via measurement.


In [ ]:
n = 3
N = 2**n

# n-qubit Hadamard transform
Hn = qt.gates.hadamard_transform(n)   # 8×8 unitary, dims [[2,2,2],[2,2,2]]

# Apply to |000>
zero_n = qt.tensor([zero]*n)
uniform = Hn * zero_n

print(f"H⊗{n}|0^{n}⟩ amplitudes:")
print(np.round(uniform.full().flatten(), 4))
print(f"All amplitudes equal 1/√{N} = {1/np.sqrt(N):.4f}:",
      np.allclose(uniform.full().flatten(), 1/np.sqrt(N)))

# Verify unitarity
print("\nH⊗n is unitary:", np.allclose((Hn.dag() * Hn).full(), np.eye(N)))

# Apply to |101>
x = qt.tensor(one, zero, one)
result = Hn * x
print(f"\nH⊗3|101⟩ amplitudes (should be ±1/√8):")
expected = np.array([((-1)**(1*int(b[0])+0*int(b[1])+1*int(b[2])))/np.sqrt(N)
                     for b in [format(k,'03b') for k in range(N)]])
print("match:", np.allclose(result.full().flatten(), expected))


### 1.5 Entanglement and Bell states

A 2-qubit state $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$ is **separable** if $|\psi\rangle = |\phi\rangle\otimes|\chi\rangle$; otherwise it is **entangled**.

The four **Bell states** form a maximally entangled orthonormal basis:
$$|\Phi^\pm\rangle = \tfrac{1}{\sqrt{2}}(|00\rangle \pm |11\rangle), \quad
  |\Psi^\pm\rangle = \tfrac{1}{\sqrt{2}}(|01\rangle \pm |10\rangle)$$

**Preparation:** $\mathsf{CNOT}\cdot(\mathsf{H}\otimes\mathbb{I})|00\rangle = |\Phi^+\rangle$.

**Schmidt rank:** reshape the coefficient tensor as a matrix $M_{ij}$. The Schmidt rank is $\mathrm{rank}(M)$ — equals 1 iff separable.


In [ ]:
# ── Prepare |Φ+> ─────────────────────────────────────────────────────────────
ket00 = qt.tensor(zero, zero)
HI    = qt.tensor(H, I)
Phi_plus = CNOT * HI * ket00

print("|Φ+⟩ =", np.round(Phi_plus.full().flatten(), 4))
print("Norm:", np.round(float(Phi_plus.norm()), 4))

# ── Schmidt decomposition via SVD ─────────────────────────────────────────────
def schmidt_rank(psi_2qubit):
    """Return SVD singular values for a 2-qubit state."""
    M = psi_2qubit.full().reshape(2, 2)   # row = qubit A, col = qubit B
    sv = np.linalg.svd(M, compute_uv=False)
    return sv

sv = schmidt_rank(Phi_plus)
print("\n|Φ+⟩ Schmidt values:", np.round(sv, 4), "-> rank =", np.sum(sv > 1e-9))

# Compare with a separable state
sep = qt.tensor(plus, plus)
sv_sep = schmidt_rank(sep)
print("|+>|+⟩  Schmidt values:", np.round(sv_sep, 4), "-> rank =", np.sum(sv_sep > 1e-9))

# Partial trace of |Φ+><Φ+|
rho = qt.ket2dm(Phi_plus)
rho_A = qt.ptrace(rho, 0)     # trace out qubit B
print("\nReduced state of qubit A in |Φ+⟩ (should be maximally mixed I/2):")
print(np.round(rho_A.full(), 3))


### Exercise 1.5 — Bell state preparation and Schmidt decomposition

**(a)** Construct circuits that prepare $|\Phi^-\rangle$, $|\Psi^+\rangle$, $|\Psi^-\rangle$ from $|00\rangle$ using $\mathsf{H}$, $\mathsf{X}$, and $\mathsf{CNOT}$.

**(b)** Verify all four Bell states are orthonormal.

**(c)** For $|\psi\rangle = \frac{1}{2}(|00\rangle+|01\rangle+|10\rangle+|11\rangle)$, compute the Schmidt rank.


In [ ]:
# YOUR CODE HERE

# (a) Prepare Phi_minus, Psi_plus, Psi_minus
# Hint: Z ⊗ I before or after CNOT·(H⊗I) modifies relative phases; X ⊗ I flips qubits

# Phi_minus = ...
# Psi_plus  = ...
# Psi_minus = ...

# (b) Check all four Bell states are orthonormal
# bell_states = [Phi_plus, Phi_minus, Psi_plus, Psi_minus]
# gram = np.array([[...]])

# (c) Schmidt rank of uniform superposition
# uniform_2 = qt.tensor(plus, plus)  # this is separable -- try a different state


In [ ]:
#@title Solution — Exercise 1.5  {display-mode: "form"}
XI = qt.tensor(X, I)
ZI = qt.tensor(Z, I)

Phi_minus = ZI * Phi_plus           # apply Z to first qubit
Psi_plus  = XI * Phi_plus           # flip first qubit
Psi_minus = XI * ZI * Phi_plus      # flip and phase

bell = [Phi_plus, Phi_minus, Psi_plus, Psi_minus]
labels = ["|Φ+⟩", "|Φ-⟩", "|Ψ+⟩", "|Ψ-⟩"]
for b, l in zip(bell, labels):
    print(f"{l} =", np.round(b.full().flatten(), 3))

# (b) Gram matrix should be identity
gram = np.array([[(b1.dag()*b2) for b2 in bell] for b1 in bell], dtype=complex)
print("\nGram matrix (should be I):")
print(np.round(np.abs(gram), 3))

# (c) Uniform superposition |ψ> = (1/2)(|00>+|01>+|10>+|11>)
psi_c = (qt.tensor(zero,zero)+qt.tensor(zero,one)+qt.tensor(one,zero)+qt.tensor(one,one)).unit()
sv_c = np.linalg.svd(psi_c.full().reshape(2,2), compute_uv=False)
print("\n(c) Schmidt values:", np.round(sv_c, 4), "-> rank =", np.sum(sv_c > 1e-9))
# This state equals |+>|+>, so it IS separable (rank 1)
print("    psi = |+>⊗|+>?", np.allclose(psi_c.full(), qt.tensor(plus,plus).full()))


---
## Summary

| Topic | Key result |
|-------|-----------|
| Quantum states | Unit vectors in $\mathbb{C}^{2^n}$; probabilities from Born rule $p_k = |\langle k|\psi\rangle|^2$ |
| Gates | Unitary matrices; Paulis satisfy $P^2=I$; $\mathsf{H}$ converts $Z$-eigenbasis to $X$-eigenbasis |
| Bloch sphere | Every single-qubit state is $\cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$; unitaries act as rotations |
| Hadamard transform | $\mathsf{H}^{\otimes n}|0^n\rangle = \frac{1}{\sqrt{2^n}}\sum_z|z\rangle$; basis for quantum parallelism |
| Entanglement | Schmidt rank $> 1$; Bell states are maximally entangled two-qubit states |

**Next:** Notebook 2 introduces the Quantum Fourier Transform (§2).